# MANTA Results Analysis
This notebook loads model outputs, computes statistical tests, and generates paper figures.

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

from manta.evaluation.metrics import bootstrap_confidence_interval, compute_all_metrics, mcnemar_test
from manta.evaluation.visualizer import (
    plot_ablation_heatmap,
    plot_calibration_curves,
    plot_per_planet_size_performance,
    plot_roc_curves,
)

## Step 1: Load prediction artifacts
Expected input files include y_true, MANTA probabilities, and AstroNet probabilities.

In [ ]:
artifacts_dir = Path('outputs/evaluation_artifacts')
y_true = np.load(artifacts_dir / 'y_true.npy')
manta_pred = np.load(artifacts_dir / 'manta_pred.npy')
astronet_pred = np.load(artifacts_dir / 'astronet_pred.npy')

print('Loaded arrays:', y_true.shape, manta_pred.shape, astronet_pred.shape)

## Step 2: Compute benchmark metrics
We compute the full metric set for both models and persist JSON summaries.

In [ ]:
manta_metrics = compute_all_metrics(y_true, manta_pred, threshold=0.5)
astro_metrics = compute_all_metrics(y_true, astronet_pred, threshold=0.5)

print('MANTA metrics:', manta_metrics)
print('AstroNet metrics:', astro_metrics)

out_dir = Path('outputs/results_analysis')
out_dir.mkdir(parents=True, exist_ok=True)
(out_dir / 'manta_metrics.json').write_text(json.dumps(manta_metrics, indent=2), encoding='utf-8')
(out_dir / 'astronet_metrics.json').write_text(json.dumps(astro_metrics, indent=2), encoding='utf-8')

## Step 3: Statistical significance tests
McNemar compares paired classification errors; bootstrap estimates uncertainty intervals.

In [ ]:
manta_err = (manta_pred >= 0.5).astype(int) != y_true.astype(int)
astro_err = (astronet_pred >= 0.5).astype(int) != y_true.astype(int)
mc = mcnemar_test(manta_err, astro_err)
print('McNemar:', mc)

ci_auc = bootstrap_confidence_interval(
    y_true=y_true,
    y_pred=manta_pred,
    metric_fn=lambda yt, yp: compute_all_metrics(yt, yp, threshold=0.5)['auc_roc'],
    n_bootstrap=1000,
    ci=0.95,
)
print('MANTA AUC bootstrap CI:', ci_auc)

## Step 4: Generate paper figures
This cell generates ROC and calibration figures; optional ablation/planet-size plots are generated when tables are available.

In [ ]:
fig_dir = Path('outputs/figures')
fig_dir.mkdir(parents=True, exist_ok=True)

plot_roc_curves(
    {'MANTA': {'y_true': y_true, 'y_pred': manta_pred}, 'AstroNet': {'y_true': y_true, 'y_pred': astronet_pred}},
    output_dir=fig_dir,
)

plot_calibration_curves(y_true, {'MANTA': manta_pred, 'AstroNet': astronet_pred}, output_dir=fig_dir)

ablation_csv = Path('outputs/ablation/ablation_results.csv')
if ablation_csv.exists():
    ablation_df = pd.read_csv(ablation_csv)
    plot_ablation_heatmap(ablation_df, output_dir=fig_dir)

planet_breakdown_csv = Path('outputs/results_analysis/per_planet_size.csv')
if planet_breakdown_csv.exists():
    breakdown_df = pd.read_csv(planet_breakdown_csv)
    plot_per_planet_size_performance(breakdown_df, output_dir=fig_dir)

print('Figure generation complete in', fig_dir)